In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

RUTA_CRUDOS = Path("../datos/crudos")
RUTA_PROCESADOS = Path("../datos/procesados")
RUTA_DB = RUTA_PROCESADOS / "riesgo_crediticio.db"

conn = sqlite3.connect(RUTA_DB)

In [2]:
df_app = pd.read_csv(RUTA_PROCESADOS / "application_train_limpio.csv")
df_app.to_sql("application", conn, if_exists="replace", index=False)
print(df_app.shape)

(307511, 133)


In [3]:
def cargar_csv_a_sqlite(ruta_csv, nombre_tabla, conn, tamano_bloque=200_000):
    primera = True
    total = 0
    for bloque in pd.read_csv(ruta_csv, chunksize=tamano_bloque):
        modo = "replace" if primera else "append"
        bloque.to_sql(nombre_tabla, conn, if_exists=modo, index=False)
        primera = False
        total += len(bloque)
        print(f"{nombre_tabla}: {total:,} filas cargadas")

cargar_csv_a_sqlite(RUTA_CRUDOS / "bureau.csv", "bureau", conn)
cargar_csv_a_sqlite(RUTA_CRUDOS / "previous_application.csv", "previous_application", conn)

bureau: 200,000 filas cargadas
bureau: 400,000 filas cargadas
bureau: 600,000 filas cargadas
bureau: 800,000 filas cargadas
bureau: 1,000,000 filas cargadas
bureau: 1,200,000 filas cargadas
bureau: 1,400,000 filas cargadas
bureau: 1,600,000 filas cargadas
bureau: 1,716,428 filas cargadas
previous_application: 200,000 filas cargadas
previous_application: 400,000 filas cargadas
previous_application: 600,000 filas cargadas
previous_application: 800,000 filas cargadas
previous_application: 1,000,000 filas cargadas
previous_application: 1,200,000 filas cargadas
previous_application: 1,400,000 filas cargadas
previous_application: 1,600,000 filas cargadas
previous_application: 1,670,214 filas cargadas


In [4]:
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn))

for tabla in ["application", "bureau", "previous_application"]:
    print(tabla, pd.read_sql(f"SELECT COUNT(*) AS filas FROM {tabla}", conn).iloc[0, 0])

                   name
0           application
1                bureau
2  previous_application
application 307511
bureau 1716428
previous_application 1670214


In [5]:
consulta = """
SELECT
    COUNT(*)            AS total_clientes,
    SUM(TARGET)         AS incumplieron,
    ROUND(AVG(TARGET) * 100, 2) AS tasa_incumplimiento_pct
FROM application
"""
pd.read_sql(consulta, conn)

,total_clientes,incumplieron,tasa_incumplimiento_pct
0,307511,24825,8.07


In [6]:
consulta = """
SELECT
    NAME_CONTRACT_TYPE,
    COUNT(*) AS clientes,
    ROUND(AVG(TARGET) * 100, 2) AS tasa_incumplimiento_pct
FROM application
GROUP BY NAME_CONTRACT_TYPE
ORDER BY tasa_incumplimiento_pct DESC
"""
pd.read_sql(consulta, conn)

,NAME_CONTRACT_TYPE,clientes,tasa_incumplimiento_pct
0,Cash loans,278232,8.35
1,Revolving loans,29279,5.48


In [11]:
consulta = """
SELECT
    NAME_INCOME_TYPE,
    COUNT(*) AS clientes,
    ROUND(AVG(TARGET) * 100, 2) AS tasa_incumplimiento_pct
FROM application
WHERE CODE_GENDER IS NOT NULL
GROUP BY NAME_INCOME_TYPE
HAVING COUNT(*) >= 1000
ORDER BY tasa_incumplimiento_pct DESC
"""
pd.read_sql(consulta, conn)

,NAME_INCOME_TYPE,clientes,tasa_incumplimiento_pct
0,Working,158771,9.59
1,Commercial associate,71616,7.48
2,State servant,21703,5.75
3,Pensioner,55362,5.39


In [13]:
consulta = """ 
SELECT
    RANGO_EDAD,
    COUNT(*) AS clientes,
    ROUND(AVG(TARGET) * 100, 2) AS tasa_incumplimiento_pct
FROM application
GROUP BY RANGO_EDAD
ORDER BY RANGO_EDAD 
 """
pd.read_sql(consulta, conn)

,RANGO_EDAD,clientes,tasa_incumplimiento_pct
0,20-29,45000,11.46
1,30-39,82299,9.58
2,40-49,76543,7.66
3,50-59,68074,6.13
4,60-69,35595,4.92


In [14]:
consulta = """
SELECT
    COUNT(*)                      AS filas,
    COUNT(DISTINCT SK_ID_CURR)    AS clientes_distintos,
    ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT SK_ID_CURR), 2) AS creditos_por_cliente
FROM bureau
"""
pd.read_sql(consulta, conn)

,filas,clientes_distintos,creditos_por_cliente
0,1716428,305811,5.61


In [15]:
consulta = """
SELECT CREDIT_ACTIVE, COUNT(*) AS creditos
FROM bureau
GROUP BY CREDIT_ACTIVE
ORDER BY creditos DESC
"""
pd.read_sql(consulta, conn)

,CREDIT_ACTIVE,creditos
0,Closed,1079273
1,Active,630607
2,Sold,6527
3,Bad debt,21


In [16]:
conn.execute("DROP TABLE IF EXISTS bureau_resumen")

consulta = """
CREATE TABLE bureau_resumen AS
SELECT
    SK_ID_CURR,
    COUNT(*)                                                AS BUREAU_NUM_CREDITOS,
    SUM(CASE WHEN CREDIT_ACTIVE = 'Active' THEN 1 ELSE 0 END) AS BUREAU_NUM_ACTIVOS,
    SUM(CASE WHEN CREDIT_DAY_OVERDUE > 0 THEN 1 ELSE 0 END)   AS BUREAU_NUM_CON_ATRASO,
    MAX(CREDIT_DAY_OVERDUE)                                 AS BUREAU_MAX_DIAS_ATRASO,
    SUM(AMT_CREDIT_SUM)                                     AS BUREAU_MONTO_TOTAL,
    SUM(AMT_CREDIT_SUM_DEBT)                                AS BUREAU_DEUDA_TOTAL
FROM bureau
GROUP BY SK_ID_CURR
"""
conn.execute(consulta)
conn.commit()

In [ ]:
pd.read_sql("SELECT COUNT(*) AS filas, COUNT(DISTINCT SK_ID_CURR) AS clientes FROM bureau_resumen", conn)

,filas,clientes
0,305811,305811


In [18]:
conn.execute("DROP TABLE IF EXISTS base_unida")

consulta = """
CREATE TABLE base_unida AS
SELECT
    a.*,
    b.BUREAU_NUM_CREDITOS,
    b.BUREAU_NUM_ACTIVOS,
    b.BUREAU_NUM_CON_ATRASO,
    b.BUREAU_MAX_DIAS_ATRASO,
    b.BUREAU_MONTO_TOTAL,
    b.BUREAU_DEUDA_TOTAL,
    CASE WHEN b.SK_ID_CURR IS NULL THEN 1 ELSE 0 END AS BUREAU_SIN_HISTORIAL
FROM application a
LEFT JOIN bureau_resumen b
    ON a.SK_ID_CURR = b.SK_ID_CURR
"""
conn.execute(consulta)
conn.commit()

In [19]:
consulta = """
SELECT
    COUNT(*) AS filas,
    COUNT(DISTINCT SK_ID_CURR) AS clientes,
    SUM(BUREAU_SIN_HISTORIAL) AS sin_historial
FROM base_unida
"""
pd.read_sql(consulta, conn)

,filas,clientes,sin_historial
0,307511,307511,44020


In [21]:
consulta = """
SELECT
    BUREAU_SIN_HISTORIAL,
    COUNT(*) AS clientes,
    ROUND(AVG(TARGET) * 100, 2) AS tasa_incumplimiento_pct
FROM base_unida
GROUP BY BUREAU_SIN_HISTORIAL
"""
pd.read_sql(consulta, conn)

,BUREAU_SIN_HISTORIAL,clientes,tasa_incumplimiento_pct
0,0,263491,7.73
1,1,44020,10.12


In [20]:
consulta = """
SELECT COUNT(*) AS clientes_bureau_fuera_de_application
FROM bureau_resumen
WHERE SK_ID_CURR NOT IN (SELECT SK_ID_CURR FROM application)
"""
pd.read_sql(consulta, conn)

,clientes_bureau_fuera_de_application
0,42320


In [22]:
consulta = """
SELECT NAME_CONTRACT_STATUS, COUNT(*) AS solicitudes
FROM previous_application
GROUP BY NAME_CONTRACT_STATUS
ORDER BY solicitudes DESC
"""
pd.read_sql(consulta, conn)

,NAME_CONTRACT_STATUS,solicitudes
0,Approved,1036781
1,Canceled,316319
2,Refused,290678
3,Unused offer,26436


In [24]:
conn.execute("DROP TABLE IF EXISTS previous_resumen")

consulta = """
CREATE TABLE previous_resumen AS
SELECT
    SK_ID_CURR,
    COUNT(*)                                                          AS PREV_NUM_SOLICITUDES,
    SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Approved' THEN 1 ELSE 0 END) AS PREV_NUM_APROBADAS,
    SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Refused'  THEN 1 ELSE 0 END) AS PREV_NUM_RECHAZADAS,
    AVG(AMT_APPLICATION)                                              AS PREV_MONTO_PROMEDIO_SOLICITADO
FROM previous_application
GROUP BY SK_ID_CURR
"""
conn.execute(consulta)
conn.commit()

In [25]:
pd.read_sql("SELECT COUNT(*) AS filas, COUNT(DISTINCT SK_ID_CURR) AS clientes FROM previous_resumen", conn)

,filas,clientes
0,338857,338857


In [26]:
consulta = """
SELECT COUNT(*) AS clientes_sin_historial_previo
FROM application
WHERE SK_ID_CURR NOT IN (SELECT SK_ID_CURR FROM previous_resumen)
"""
pd.read_sql(consulta, conn)

,clientes_sin_historial_previo
0,16454


In [27]:
consulta = """
SELECT COUNT(*) AS clientes_previous_fuera_de_application
FROM previous_resumen
WHERE SK_ID_CURR NOT IN (SELECT SK_ID_CURR FROM application)
"""
pd.read_sql(consulta, conn)

,clientes_previous_fuera_de_application
0,47800


In [28]:
conn.execute("DROP TABLE IF EXISTS base_completa")

consulta = """
CREATE TABLE base_completa AS
SELECT
    u.*,
    p.PREV_NUM_SOLICITUDES,
    p.PREV_NUM_APROBADAS,
    p.PREV_NUM_RECHAZADAS,
    p.PREV_MONTO_PROMEDIO_SOLICITADO,
    CASE WHEN p.SK_ID_CURR IS NULL THEN 1 ELSE 0 END AS PREV_SIN_HISTORIAL
FROM base_unida u
LEFT JOIN previous_resumen p
    ON u.SK_ID_CURR = p.SK_ID_CURR
"""
conn.execute(consulta)
conn.commit()

In [29]:
consulta = """
SELECT
    PREV_SIN_HISTORIAL,
    COUNT(*) AS clientes,
    ROUND(AVG(TARGET) * 100, 2) AS tasa_incumplimiento_pct
FROM base_completa
GROUP BY PREV_SIN_HISTORIAL
"""
pd.read_sql(consulta, conn)

,PREV_SIN_HISTORIAL,clientes,tasa_incumplimiento_pct
0,0,291057,8.19
1,1,16454,5.96


In [30]:
consulta = """
SELECT
    NAME_EDUCATION_TYPE,
    COUNT(*) AS clientes,
    ROUND(AVG(TARGET) * 100, 2) AS tasa_incumplimiento_pct
FROM base_completa
GROUP BY NAME_EDUCATION_TYPE
HAVING COUNT(*) >= 1000
ORDER BY tasa_incumplimiento_pct DESC
"""
pd.read_sql(consulta, conn)

,NAME_EDUCATION_TYPE,clientes,tasa_incumplimiento_pct
0,Lower secondary,3816,10.93
1,Secondary / secondary special,218391,8.94
2,Incomplete higher,10277,8.48
3,Higher education,74863,5.36


In [31]:
consulta = """
SELECT
    RANGO_EDAD,
    NAME_CONTRACT_TYPE,
    COUNT(*) AS clientes,
    ROUND(AVG(TARGET) * 100, 2) AS tasa_incumplimiento_pct
FROM base_completa
GROUP BY RANGO_EDAD, NAME_CONTRACT_TYPE
HAVING COUNT(*) >= 1000
ORDER BY RANGO_EDAD, NAME_CONTRACT_TYPE
"""
pd.read_sql(consulta, conn)

,RANGO_EDAD,NAME_CONTRACT_TYPE,clientes,tasa_incumplimiento_pct
0,20-29,Cash loans,38289,12.05
1,20-29,Revolving loans,6711,8.05
2,30-39,Cash loans,74340,10.01
3,30-39,Revolving loans,7959,5.60
4,40-49,Cash loans,69441,7.95
5,40-49,Revolving loans,7102,4.79
6,50-59,Cash loans,62605,6.33
7,50-59,Revolving loans,5469,3.80
8,60-69,Cash loans,33557,5.01
9,60-69,Revolving loans,2038,3.43


In [32]:
consulta = """
SELECT
    CASE
        WHEN PREV_NUM_RECHAZADAS IS NULL THEN 'Sin historial'
        WHEN PREV_NUM_RECHAZADAS = 0     THEN '0 rechazos'
        WHEN PREV_NUM_RECHAZADAS = 1     THEN '1 rechazo'
        ELSE '2 o más rechazos'
    END AS rechazos_previos,
    COUNT(*) AS clientes,
    ROUND(AVG(TARGET) * 100, 2) AS tasa_incumplimiento_pct
FROM base_completa
GROUP BY rechazos_previos
ORDER BY tasa_incumplimiento_pct DESC
"""
pd.read_sql(consulta, conn)

,rechazos_previos,clientes,tasa_incumplimiento_pct
0,2 o más rechazos,53760,11.61
1,1 rechazo,46534,8.83
2,0 rechazos,190763,7.07
3,Sin historial,16454,5.96


In [33]:
df_base = pd.read_sql("SELECT * FROM base_completa", conn)
df_base.to_csv(RUTA_PROCESADOS / "base_completa.csv", index=False)
print(df_base.shape)

(307511, 145)
